In [ ]:
# ============================================================
# PARAMETERS (papermill injects these)
# ============================================================

config_path = None
run_dir = None

In [ ]:
# ============================================================
# IMPORTS
# ============================================================

import json
import yaml
import random
import numpy as np
import pandas as pd
import torch
import os

from collections import Counter

from pathlib import Path

from scripts.set_seed import set_seed

from src.model_factory import build_model

from src.dataset import MSADataset, CLASS_MAP

from src.embedder import MSAEmbedder, ESM2Embedder

from src.training import evaluate_split

from src.explainibilty import (
    make_zero_baseline,
    explain_predictions,
    print_results,
    visualize_sequence_explanations,
    visualize_attention_explanations,
    compute_attention_weights,
    compute_saliency,
    results_to_json_compatible,
    map_ecs_explanations_to_full_sequence,
    map_ecs_attention_to_full_sequence,
)

In [ ]:
# ============================================================
# LOAD CONFIG, SEED, AND SAVED MODEL
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

with open(config_path, "r") as f:
    cfg = yaml.safe_load(f)

RUN_DIR = Path(run_dir)
RUN_DIR.mkdir(parents=True, exist_ok=True)

set_seed(cfg["experiment"]["seed"])

embedder_cfg = cfg.get("embedder", {}) or {}
embedder_name = (embedder_cfg.get("name") or "msa_transformer").lower()
embedding_dim = 640 if embedder_name == "esm2" else 768

model_init_args = dict(cfg.get("model", {}).get("init_args", {}))
model_init_args.setdefault("embedding_dim", embedding_dim)
cfg.setdefault("model", {}).setdefault("init_args", {}).update(model_init_args)

model = build_model(cfg, instantiate=True, device=device)
state = torch.load(RUN_DIR / "final_model.pt", map_location=device)
model.load_state_dict(state)
model.eval()
print(f"Loaded model from {RUN_DIR / 'final_model.pt'}")

print(json.dumps(cfg, indent=2))

In [ ]:
# ============================================================
# UTILITY: Extract ECS regions for ECS-only models
# ============================================================


def extract_ecs_regions(sequences, ecs_regions):
    """
    Extract ECS regions from sequences.

    Args:
        sequences: List of sequence strings
        ecs_regions: List of [start, end] tuples (1-indexed, inclusive)

    Returns:
        List of concatenated ECS region substrings
    """
    if not ecs_regions:
        return sequences  # No ECS extraction needed

    ecs_seqs = []
    for seq in sequences:
        # Convert 1-indexed regions to 0-indexed slices
        ecs_parts = [seq[start - 1 : end] for start, end in ecs_regions]
        ecs_seq = "".join(ecs_parts)
        ecs_seqs.append(ecs_seq)
    return ecs_seqs


# Check if we're in ECS-only mode
is_ecs_only = cfg["data"]["dataset_type"] == "ecs_only"

# Define ECS regions for each test set
# test_data1: always use hardcoded regions (fixed reference dataset)
ecs_regions_test1 = [[28, 81], [139, 164]] if is_ecs_only else []

# test_data2: use regions from config
ecs_regions_test2 = cfg["data"].get("ecs_only_regions", []) if is_ecs_only else []

# Store original sequence length for full-sequence visualization
original_seq_len_1 = None  # Will be set after loading test_data1
original_seq_len_2 = None  # Will be set after loading test_data2

if is_ecs_only:
    print(f"ECS-only mode enabled.")
    print(f"  test_data1 (fixed reference): {ecs_regions_test1}")
    print(f"  test_data2 (from config): {ecs_regions_test2}")
else:
    print("Full-sequence mode (or no ECS regions specified)")

In [ ]:

# ============================================================
# LOAD AND EMBED TEST DATA
# ============================================================

# Check if reference sequences exist in RUN_DIR
ref_seqs_path = RUN_DIR / "reference_sequences.fasta"
if ref_seqs_path.exists():
    print(f"Loading reference sequences from {ref_seqs_path} to be considered along test sequences for embedding extraction.")
    reference_msa = str(ref_seqs_path)
else:
    print(f"No reference sequences found at {ref_seqs_path}. Test sequences will be embedded without reference MSA context.")
    reference_msa = None

# Inference on a fixed set of sequences
test_data1 = MSADataset(
    ["/content/drive/MyDrive/Thesis data/MSAs/test_set.fasta"],
    [-1],
    test_data=True,
    reference_msa=reference_msa,
    ecs_only=is_ecs_only,
)
test_data1_seq_len = test_data1.getSequenceLength()
test_data1_seq_ids, test_data1_seqs = test_data1.getSequences()
test_data1_mask = test_data1.getUnseenSequenceMasks()
test_data1_labels = test_data1.getLabels()

# Store original sequence length before ECS extraction
original_seq_len_1 = len(test_data1_seqs[0]) if test_data1_seqs else test_data1_seq_len
original_seqs_1 = test_data1_seqs.copy()  # Keep original for display later

# Extract ECS regions if in ECS-only mode (using hardcoded regions for test_data1)
if is_ecs_only and ecs_regions_test1:
    test_data1_seqs = extract_ecs_regions(test_data1_seqs, ecs_regions_test1)
    test_data1_seq_len = (
        len(test_data1_seqs[0]) if test_data1_seqs else test_data1_seq_len
    )

print(f"# Test sequences (test_data1): {len(test_data1_seqs)}")
print(f"Sequence length (after ECS extraction if applicable): {test_data1_seq_len}")
if is_ecs_only and ecs_regions_test1:
    print(f"Original full sequence length: {original_seq_len_1}")

# Instantiate embedder based on top-level config
embedder_cfg = cfg.get("embedder", {}) or {}
embedder_name = (embedder_cfg.get("name") or "msa_transformer").lower()

if embedder_name == "esm2":
    embedder = ESM2Embedder(device=device)
    use_msa = False
else:
    embedder = MSAEmbedder(device=device)
    use_msa = embedder_cfg.get("use_msa_mode", cfg["data"].get("use_msa_mode", True))

if use_msa:
    test_data1_embeddings = embedder.embed_msa(
        sequences=test_data1_seqs,
        seq_length=test_data1_seq_len,
        max_msa_depth=len(test_data1_seqs),
    )
else:
    test_data1_embeddings = embedder.embed_sequences_per_residue(
        sequences=test_data1_seqs, seq_length=test_data1_seq_len, batch_size=1
    )

# Check if mask is not empty — apply it consistently to all arrays
if test_data1_mask:
    mask = test_data1_mask[0]
    test_data1_embeddings = test_data1_embeddings[mask]
    test_data1_seq_ids = np.array(test_data1_seq_ids)[mask]
    test_data1_seqs = np.array(test_data1_seqs)[mask]
    original_seqs_1 = np.array(original_seqs_1)[mask]
    # Labels are over all combined_ids (reference + unseen), so mask them too
    test_data1_labels = np.array(test_data1_labels)[mask]

    print(f"Embeddings shape: {test_data1_embeddings.shape} with reference MSA context")
else:
    print(f"Embeddings shape: {test_data1_embeddings.shape} without reference MSA context")

# Run inference with final model
model.eval()
with torch.no_grad():
    logits1 = model(test_data1_embeddings.to(device))
    probs1 = torch.softmax(logits1, dim=1)
    pred1 = probs1.argmax(dim=1)

for i, cls in enumerate(pred1.cpu().numpy()):
    print(f"\n({i}) {test_data1_seq_ids[i]}:")
    if is_ecs_only and ecs_regions_test1:
        print(f"    Original sequence: {original_seqs_1[i]}")
        print(f"    ECS-only sequence: {test_data1_seqs[i]}")
    else:
        print(f"    Sequence: {test_data1_seqs[i]}")
    print(f"    Predicted class: {CLASS_MAP[cls]}, confidence={probs1[i, cls]:.3f}")
    print(f"    True class: {CLASS_MAP[test_data1_labels[i]] if test_data1_labels is not None else 'N/A'}")

if test_data1_labels is not None:
    metrics1 = evaluate_split("test_data1 Evaluation", pred1, test_data1_labels)
    eval_df = pd.DataFrame(
        {
            "expt": [cfg["experiment"]["name"]],
            "split": ["test_data1"],
            "accuracy": [metrics1["acc"]],
            "balanced_accuracy": [metrics1["bal_acc"]],
            "macro_precision": [metrics1["macro_p"]],
            "macro_recall": [metrics1["macro_r"]],
            "macro_f1": [metrics1["macro_f1"]],
        }
    )

# Save preds and metrics
if is_ecs_only and ecs_regions_test1:
    preds_df = pd.DataFrame(
        {
            "seq_id": test_data1_seq_ids,
            "sequence": (
                original_seqs_1
                if is_ecs_only and ecs_regions_test1
                else test_data1_seqs
            ),
            "ecs_only_region": test_data1_seqs,
            "predicted_class": [CLASS_MAP[cls] for cls in pred1.cpu().numpy()],
            "confidence": probs1.cpu()
            .numpy()
            .tolist(),  # confidence across all classes
        }
    )
else:
    preds_df = pd.DataFrame(
        {
            "seq_id": test_data1_seq_ids,
            "sequence": (
                original_seqs_1
                if is_ecs_only and ecs_regions_test1
                else test_data1_seqs
            ),
            "predicted_class": [CLASS_MAP[cls] for cls in pred1.cpu().numpy()],
            "confidence": probs1.cpu()
            .numpy()
            .tolist(),  # confidence across all classes
        }
    )
os.makedirs(RUN_DIR / "inference/predictions", exist_ok=True)
os.makedirs(RUN_DIR / "inference/metrics", exist_ok=True)
preds_df.to_csv(RUN_DIR / "inference/predictions/test_predictions1.csv", index=False)
eval_df.to_csv(RUN_DIR / "inference/metrics/test_metrics1.csv", index=False)


In [ ]:
# ============================================================
# EXPLAINIBILTY - IG
# ============================================================

test_data1_seq_ids_to_explain = [test_data1_seq_ids[i] for i in [0, 13, 34]]
test_data1_seqs_to_explain = [test_data1_seqs[i] for i in [0, 13, 34]]
test_data1_seqs_to_explain_original = (
    [original_seqs_1[i] for i in [0, 13, 34]]
    if is_ecs_only and ecs_regions_test1
    else test_data1_seqs_to_explain
)
test_data1_embeddings_to_explain = test_data1_embeddings[[0, 13, 34], :, :]

baseline_embedding1 = make_zero_baseline(
    test_data1_embeddings_to_explain.shape[1], embed_dim=embedder.embedding_dim
)

predicted_classes1 = pred1[[0, 13, 34]]
confidences1 = probs1[[0, 13, 34]].max(dim=1)[0]

true_classes1 = [test_data1_labels[i] for i in [0, 13, 34]] if test_data1_labels is not None else predicted_classes1

# ── Compute IG explanations ──
results1 = explain_predictions(
    model,
    test_data1_seq_ids_to_explain,
    test_data1_seqs_to_explain,
    test_data1_embeddings_to_explain,
    baseline_embedding1,
    predicted_classes1,
    confidences1,
    true_classes1,
    k=10,
    n_steps=100,
    device=device,
    run_ablation=True,
)

# Map ECS-only attributions back to full sequence for visualization (using hardcoded regions for test_data1)
if is_ecs_only and ecs_regions_test1:
    results1 = map_ecs_explanations_to_full_sequence(
        results1, ecs_regions_test1, original_seq_len_1
    )
    for sample in results1["samples"]:
        sample_idx = sample["sample_id"]
        sample["sequence"] = test_data1_seqs_to_explain_original[sample_idx]

    # Also update the display sequences for visualizations
    test_data1_seqs_to_explain = test_data1_seqs_to_explain_original

print_results(results1)

# Save explanations - convert to JSON-compatible format here
os.makedirs(
    os.path.dirname(RUN_DIR / "inference/explanations/explanations1_ig.json"),
    exist_ok=True,
)
with open(RUN_DIR / "inference/explanations/explanations1_ig.json", "w") as f:
    json.dump(results_to_json_compatible(results1), f, indent=2)

In [ ]:
visualize_sequence_explanations(
    results=results1,
    true_labels=true_classes1,
    class_names=list(CLASS_MAP.values()),
    max_sequences=5,
    save_name=RUN_DIR / "inference/explanations/explanations1_ig_viz",
)

In [ ]:
# ============================================================
# EXPLAINIBILTY - ATTENTION / SALIENCY
# ============================================================

if cfg["model"]["uses_attention"]:
    _, attention_weights1 = compute_attention_weights(
        model, test_data1_embeddings_to_explain.to(device)
    )
    save_name = RUN_DIR / "inference/explanations/explanations1_attn"
    is_saliency = False
else:
    _, attention_weights1 = compute_saliency(
        model, test_data1_embeddings_to_explain.to(device)
    )
    save_name = RUN_DIR / "inference/explanations/explanations1_saliency"
    is_saliency = True

# Map ECS-only attention/saliency scores back to full sequence (using hardcoded regions for test_data1)
if is_ecs_only and ecs_regions_test1:
    attention_weights1 = map_ecs_attention_to_full_sequence(
        attention_weights1, ecs_regions_test1, original_seq_len_1
    )

visualize_attention_explanations(
    attention_weights1,
    test_data1_seqs_to_explain,
    test_data1_seq_ids_to_explain,
    predicted_classes1,
    confidences1,
    true_classes1,
    class_names=list(CLASS_MAP.values()),
    save_name=save_name,
    is_saliency=is_saliency,
)

In [ ]:
# ============================================================
# LOAD AND EMBED TEST DATA SPECIFIED IN CONFIG
# ============================================================

if reference_msa:
    print(f"Using reference MSA context from {reference_msa} for embedding test_data2 sequences.")

# Inference on a fixed set of sequences
test_data2 = MSADataset(
    [f'/content/drive/MyDrive/Thesis data/MSAs/{cfg["evaluation"]["test_data"]}'],
    [-1],
    test_data=True,
    reference_msa=reference_msa,
    ecs_only=is_ecs_only,
)
test_data2_seq_len = test_data2.getSequenceLength()
test_data2_seq_ids, test_data2_seqs = test_data2.getSequences()
test_data2_labels = cfg["evaluation"]["test_data_labels"]
test_data2_mask = test_data2.getUnseenSequenceMasks()

# Store original sequence length before ECS extraction
original_seq_len_2 = len(test_data2_seqs[0]) if test_data2_seqs else test_data2_seq_len
original_seqs_2 = test_data2_seqs.copy()  # Keep original for display later

# Extract ECS regions if in ECS-only mode (using config-specified regions for test_data2)
if is_ecs_only and ecs_regions_test2:
    test_data2_seqs = extract_ecs_regions(test_data2_seqs, ecs_regions_test2)
    test_data2_seq_len = (
        len(test_data2_seqs[0]) if test_data2_seqs else test_data2_seq_len
    )

print(f"# Test sequences (test_data2): {len(test_data2_seqs)}")
print(f"Sequence length (after ECS extraction if applicable): {test_data2_seq_len}")
if is_ecs_only and ecs_regions_test2:
    print(f"Original full sequence length: {original_seq_len_2}")

# Embed either in MSA mode or independently depending on the config
if use_msa:
    test_data2_embeddings = embedder.embed_msa(
        sequences=test_data2_seqs,
        seq_length=test_data2_seq_len,
        max_msa_depth=len(test_data2_seqs),
    )
else:
    test_data2_embeddings = embedder.embed_sequences_per_residue(
        sequences=test_data2_seqs, seq_length=test_data2_seq_len, batch_size=1
    )

if test_data2_mask:
    mask2 = test_data2_mask[0]
    test_data2_embeddings = test_data2_embeddings[mask2]
    test_data2_seq_ids = np.array(test_data2_seq_ids)[mask2]
    test_data2_seqs = np.array(test_data2_seqs)[mask2]
    original_seqs_2 = np.array(original_seqs_2)[mask2]
    print(f"Embeddings shape: {test_data2_embeddings.shape} with reference MSA context")
else:
    print(f"Embeddings shape: {test_data2_embeddings.shape} without reference MSA context")

# Run inference with final model
model.eval()
with torch.no_grad():
    logits2 = model(test_data2_embeddings.to(device))
    probs2 = torch.softmax(logits2, dim=1)
    pred2 = probs2.argmax(dim=1)

for i, cls in enumerate(pred2.cpu().numpy()):
    print(f"\n({i}) {test_data2_seq_ids[i]}:")
    if is_ecs_only and ecs_regions_test2:
        print(f"    Original sequence: {original_seqs_2[i]}")
        print(f"    ECS-only sequence: {test_data2_seqs[i]}")
    else:
        print(f"    Sequence: {test_data2_seqs[i]}")
    print(f"    Predicted class: {CLASS_MAP[cls]}, confidence={probs2[i, cls]:.3f}")
    print(f"    True class     : {CLASS_MAP[test_data2_labels[i]]}" if test_data2_labels else "    True class     : N/A")

if test_data2_labels is not None:
    metrics2 = evaluate_split("test_data2 Evaluation", pred2, test_data2_labels)
    eval_df = pd.DataFrame(
        {
            "expt": [cfg["experiment"]["name"]],
            "split": ["test_data2"],
            "accuracy": [metrics2["acc"]],
            "balanced_accuracy": [metrics2["bal_acc"]],
            "macro_precision": [metrics2["macro_p"]],
            "macro_recall": [metrics2["macro_r"]],
            "macro_f1": [metrics2["macro_f1"]],
        }
    )

# Save preds and metrics
if is_ecs_only and ecs_regions_test2:
    preds_df = pd.DataFrame(
        {
            "seq_id": test_data2_seq_ids,
            "sequence": (
                original_seqs_2
                if is_ecs_only and ecs_regions_test2
                else test_data2_seqs
            ),
            "ecs_only_region": test_data2_seqs,
            "predicted_class": [CLASS_MAP[cls] for cls in pred2.cpu().numpy()],
            "confidence": probs2.cpu()
            .numpy()
            .tolist(),  # confidence across all classes
        }
    )
else:
    preds_df = pd.DataFrame(
        {
            "seq_id": test_data2_seq_ids,
            "sequence": (
                original_seqs_2
                if is_ecs_only and ecs_regions_test2
                else test_data2_seqs
            ),
            "predicted_class": [CLASS_MAP[cls] for cls in pred2.cpu().numpy()],
            "confidence": probs2.cpu()
            .numpy()
            .tolist(),  # confidence across all classes
        }
    )
preds_df.to_csv(RUN_DIR / "inference/predictions/test_predictions2.csv", index=False)
eval_df.to_csv(RUN_DIR / "inference/metrics/test_metrics2.csv", index=False)

In [ ]:
# ============================================================
# EXPLAINIBILTY - IG
# ============================================================

baseline_embedding2 = make_zero_baseline(
    test_data2_embeddings.shape[1], embed_dim=embedder.embedding_dim
)

confidences2 = probs2.max(1)[0]

true_classes2 =  test_data2_labels if test_data2_labels is not None else pred2  # Use true labels if available, otherwise use predicted classes for IG
test_data2_seqs_original = (
    original_seqs_2.copy() if is_ecs_only and ecs_regions_test2 else test_data2_seqs
)

# ── Compute IG explanations ──
results2 = explain_predictions(
    model,
    test_data2_seq_ids,
    test_data2_seqs,
    test_data2_embeddings,
    baseline_embedding2,
    pred2,
    confidences2,
    true_classes2,
    k=10,
    n_steps=100,
    device=device,
    run_ablation=True,
)

# Map ECS-only attributions back to full sequence for visualization (using config-based regions for test_data2)
if is_ecs_only and ecs_regions_test2:
    results2 = map_ecs_explanations_to_full_sequence(
        results2, ecs_regions_test2, original_seq_len_2
    )
    for sample in results2["samples"]:
        sample_idx = sample["sample_id"]
        sample["sequence"] = test_data2_seqs_original[sample_idx]

    # Also update the display sequences for visualizations
    test_data2_seqs = test_data2_seqs_original

print_results(results2)

# Save explanations - convert to JSON-compatible format here
os.makedirs(
    os.path.dirname(RUN_DIR / "inference/explanations/explanations2_ig.json"),
    exist_ok=True,
)
with open(RUN_DIR / "inference/explanations/explanations2_ig.json", "w") as f:
    json.dump(results_to_json_compatible(results2), f, indent=2)

In [ ]:
visualize_sequence_explanations(
    results=results2,
    true_labels=true_classes2,
    max_sequences=10,
    save_name=RUN_DIR / "inference/explanations/explanations2_ig_viz",
)

In [ ]:
# ============================================================
# EXPLAINIBILTY - ATTENTION / SALIENCY (TEST DATA 2)
# ============================================================

if cfg["model"]["uses_attention"]:
    _, attention_weights2 = compute_attention_weights(
        model, test_data2_embeddings.to(device)
    )
    save_name = RUN_DIR / "inference/explanations/explanations2_attn"
    is_saliency = False
else:
    _, attention_weights2 = compute_saliency(model, test_data2_embeddings.to(device))
    save_name = RUN_DIR / "inference/explanations/explanations2_saliency"
    is_saliency = True

# Map ECS-only attention/saliency scores back to full sequence (using config-based regions for test_data2)
if is_ecs_only and ecs_regions_test2:
    attention_weights2 = map_ecs_attention_to_full_sequence(
        attention_weights2, ecs_regions_test2, original_seq_len_2
    )

visualize_attention_explanations(
    attention_weights2,
    test_data2_seqs,
    test_data2_seq_ids,
    pred2,
    confidences2,
    true_classes2,
    class_names=list(CLASS_MAP.values()),
    save_name=save_name,
    is_saliency=is_saliency,
)